<a href="https://colab.research.google.com/github/ferdibudiutomo353-png/Tugas-Machine-Learning/blob/main/Latihan_bab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Praktikum 1: Perbandingan Strategi Imputasi

In [1]:
import numpy as np, pandas as pd
from sklearn.datasets import make_classification
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import (train_test_split,
                                     RepeatedStratifiedKFold, cross_val_score)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

X, y = make_classification(n_samples=2000, n_features=8,
                           weights=[0.95, 0.05], random_state=42)
cols = [f"fitur_{i}" for i in range(8)]
df = pd.DataFrame(X, columns=cols)

# Missing value buatan (hanya di kolom fitur)
rng = np.random.default_rng(42)
mask = rng.random(df[cols].shape) < 0.05
Xm = df[cols].mask(pd.DataFrame(mask, columns=cols))

# Split dulu, imputer di-fit hanya pada data latih (hindari data leakage)
Xtr, Xte, ytr, yte = train_test_split(Xm, y, test_size=0.2,
                                      stratify=y, random_state=42)

for nama, imp in [("Mean Imputation", SimpleImputer(strategy="mean")),
                  ("KNN Imputer", KNNImputer(n_neighbors=5))]:
    a = imp.fit_transform(Xtr)
    b = imp.transform(Xte)
    m = LogisticRegression(max_iter=1000).fit(a, ytr)
    print(nama, "F1 =", round(f1_score(yte, m.predict(b)), 3))

# Pengecekan lebih stabil: cross-validation berulang
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
for nama, imp in [("Mean", SimpleImputer(strategy="mean")),
                  ("KNN", KNNImputer(n_neighbors=5))]:
    s = cross_val_score(make_pipeline(imp, LogisticRegression(max_iter=1000)),
                        Xm, y, cv=cv, scoring="f1")
    print(f"CV F1 {nama}: {s.mean():.3f} +- {s.std():.3f}")

Mean Imputation F1 = 0.5
KNN Imputer F1 = 0.5
CV F1 Mean: 0.533 +- 0.087
CV F1 KNN: 0.524 +- 0.083


Praktikum 2: Efektivitas Penanganan Imbalanced Data (Praktikum 3.7)

In [2]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Imputasi (KNN) -> data latih/uji bersih
imp = KNNImputer(n_neighbors=5)
X_train = pd.DataFrame(imp.fit_transform(Xtr), columns=cols)
X_test = pd.DataFrame(imp.transform(Xte), columns=cols)
y_train, y_test = pd.Series(ytr), pd.Series(yte)

f1 = {}

# Baseline
model_base = LogisticRegression(max_iter=1000).fit(X_train, y_train)
f1["Baseline"] = f1_score(y_test, model_base.predict(X_test))

# SMOTE
X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_train, y_train)
model_smote = LogisticRegression(max_iter=1000).fit(X_sm, y_sm)
f1["SMOTE"] = f1_score(y_test, model_smote.predict(X_test))

# Undersampling (Praktikum 3.7)
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
model_rus = LogisticRegression(max_iter=1000).fit(X_train_rus, y_train_rus)
f1["Undersampling"] = f1_score(y_test, model_rus.predict(X_test))

# Class weight
model_cw = LogisticRegression(max_iter=1000,
                              class_weight="balanced").fit(X_train, y_train)
f1["Class weight"] = f1_score(y_test, model_cw.predict(X_test))

for k, v in f1.items():
    print(f"{k:15s} F1 = {v:.3f}")

Baseline        F1 = 0.500
SMOTE           F1 = 0.420
Undersampling   F1 = 0.456
Class weight    F1 = 0.395
